Actividad #2 - Calidad, drift y anomalías

# 0 Libraries & Data Load

In [ ]:
!pip install mlcroissant
!pip install great_expectations
!pip install evidently

In [ ]:


import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.inspection import permutation_importance

# import shap

# Configuración visual
sns.set_theme(style="whitegrid")

# Desactivar la notación científica en Pandas (muestra hasta 2 decimales normales)
pd.set_option('display.float_format', lambda x: '%.2f' % x)


In [ ]:


# Fetch the Croissant JSON-LD
croissant_dataset = mlc.Dataset('https://www.kaggle.com/datasets/arnabchaki/data-science-salaries-2023/croissant/download')

# Check what record sets are in the dataset
record_sets = croissant_dataset.metadata.record_sets
print(record_sets)

# Fetch the records and put them in a DataFrame
data = pd.DataFrame(croissant_dataset.records(record_set=record_sets[0].uuid))# Esto elimina el prefijo incluyendo la barra "/" o "\" en cualquier dirección
data.columns = data.columns.str.replace(r'^/?ds_salaries\.csv/?', '', regex=True)


# Seleccionamos todas las columnas que son de tipo texto (object)
columnas_texto = data.select_dtypes(include=['object']).columns

# Recorremos esas columnas y decodificamos los bytes a texto normal (utf-8)
for col in columnas_texto:
    data[col] = data[col].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

# Volvemos a ver cómo quedó
data.head()


print(data.info())
display(data.head())



In [ ]:
data_dev = data.query("work_year <= 2021")
print(data_dev["work_year"].value_counts())

data_prod = data.query("work_year > 2021")
print(data_prod["work_year"].value_counts())


# 1 - Calidad y Drift

##### 1.a Elegir 3-4 variables relevantes (por ej. salary_in_usd, experience_level, remote_ratio, company_size) y aplicar al menos 3 chequeos de calidad a elección

In [ ]:


import great_expectations as gx

# 1. Configuración del contexto y batch para datos Actuales (PROD: 2022-2023)
context = gx.get_context()
data_source = context.data_sources.add_pandas("pandas_source")
asset = data_source.add_dataframe_asset(name="asset_salarios_prod")
batch_def = asset.add_batch_definition_whole_dataframe("batch_def_prod")
batch_prod = batch_def.get_batch(batch_parameters={"dataframe": data_prod})

# 2. Chequeos de calidad sobre 4 variables relevantes

# a. Completitud (salary_in_usd): Verificamos que no existan valores nulos
res_completitud = batch_prod.validate(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="salary_in_usd")
)
print(f"Completitud (salary_in_usd): {res_completitud.success}")

# b. Validez de Rango (remote_ratio): Asegurar que los valores sean lógicos (0, 50, 100)
res_consistencia = batch_prod.validate(
    gx.expectations.ExpectColumnValuesToBeInSet(column="remote_ratio", value_set=[0, 50, 100])
)
print(f"Validez Dominio (remote_ratio): {res_consistencia.success}")

# c. Validez de Categoría (company_size): Confirmar que respeta las categorías esperadas
res_categoria_size = batch_prod.validate(
    gx.expectations.ExpectColumnValuesToBeInSet(column="company_size", value_set=["S", "M", "L"])
)
print(f"Validez Categoría (company_size): {res_categoria_size.success}")

# d. Validez de Categoría (experience_level): Niveles estándar
res_categoria_exp = batch_prod.validate(
    gx.expectations.ExpectColumnValuesToBeInSet(column="experience_level", value_set=["EN", "MI", "SE", "EX"])
)
print(f"Validez Categoría (experience_level): {res_categoria_exp.success}")

##### 1.b Sobre las mismas variables, calcular PSI comparando la distribución de referencia vs. la actual

In [ ]:
import sys
import site

# Agregar de forma oficial la ruta de paquetes de Google Colab al entorno de búsqueda
site.addsitedir('/usr/local/lib/python3.13/dist-packages')

import pandas as pd

# Intentar las diferentes variantes de importación según la versión de evidently
try:
    from evidently.metric_preset import DataDriftPreset
except ModuleNotFoundError:
    try:
        from evidently.metric_presets import DataDriftPreset
    except ModuleNotFoundError:
        from evidently.presets import DataDriftPreset

from evidently.report import Report
from evidently import ColumnMapping

# 1. Variables de evaluación
variables_evaluacion = ["salary_in_usd", "remote_ratio", "company_size", "experience_level"]

ref_df = data_dev[variables_evaluacion].copy()
prod_df = data_prod[variables_evaluacion].copy()

# 2. Mapeo explícito de tipos
column_mapping = ColumnMapping(
    numerical_features=["salary_in_usd"],
    categorical_features=["remote_ratio", "company_size", "experience_level"]
)

# 3. Configuración del reporte con PSI (umbral estándar = 0.2 para drift significativo)
drift_report = Report(metrics=[
    DataDriftPreset(stattest="psi", stattest_threshold=0.2)
])

# 4. Ejecución pasando el column_mapping
drift_report.run(
    reference_data=ref_df,
    current_data=prod_df,
    column_mapping=column_mapping
)

# 5. Visualización interactiva
drift_report.show(mode='inline')